# Energy Poverty Analysis
Classification of households using 18 validation checks.

In [22]:
def run_safe(callback, label):
    try:
        print(f"Executing: {label}")
        callback()
        print(f"Completed: {label}\n")
    except Exception as e:
        print(f"ERROR in {label}: {e}")


In [23]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')


In [24]:
def data_workflow():
    global df, X, y, X_train, X_test, y_train, y_test
    df = pd.read_csv("Cleaned_dataset_energy_poverty.csv")
    df["energy_score"] = (
        (df["Electricity"] == 0) * 3 +
        (df["hrs_elect"] < 8) * 2 +
        (df["cfuel"] == 1) * 2 +
        (df["avl_lpg"] == 0) * 1 +
        (df["ipollut"] == 1) * 1
    )
    df["Energy_Poverty"] = (df["energy_score"] >= 3).astype(int)
    drop_list = ["hh", "Energy_Poverty", "energy_score", "Electricity", "hrs_elect", "cfuel", "avl_lpg", "ipollut"]
    X = df.drop([c for c in drop_list if c in df.columns], axis=1)
    y = df["Energy_Poverty"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

run_safe(data_workflow, "Data Cleaning and Split")


Executing: Data Cleaning and Split
Completed: Data Cleaning and Split



In [25]:
def model_training():
    global results
    models = {
        "Logistic Regression": LogisticRegression(max_iter=500),
        "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42),
        "SVM": SVC(kernel='linear', random_state=42)
    }
    results = []
    for name, m in models.items():
        m.fit(X_train, y_train)
        acc = accuracy_score(y_test, m.predict(X_test))
        results.append((name, acc))
        print(f"{name}: {acc:.2%}")

run_safe(model_training, "ML Model Training")


Executing: ML Model Training
Logistic Regression: 60.00%
Decision Tree: 60.67%
Random Forest: 66.67%
SVM: 56.00%
Completed: ML Model Training



In [26]:

def stress_testing_suite():
    print("====================================================")
    print(" SECTION 15: AUTOMATED QA STRESS TESTS ")
    print("====================================================\n")
    
    total = 18
    passed = 0
    
    def check(tid, name, cond, detail=""):
        nonlocal passed
        res = "[Pass]" if cond else "[Fail]"
        if cond: passed += 1
        print(f"{res} Test {tid:02d}: {name} ({detail})")

    check(1, "No Missing Features", X.isnull().sum().sum() == 0, "0 nulls")
    check(2, "No Missing Target", y.isnull().sum() == 0, "0 nulls")
    check(3, "Rows > 400", len(X) > 400, f"{len(X)} rows")
    check(4, "Features > 20", X.shape[1] > 20, f"{X.shape[1]} features")
    check(5, "Target Binary", list(np.unique(y)) == [0, 1], "0,1")
    check(6, "Minority >= 15%", (y.value_counts(normalize=True).min()) >= 0.15, f"{y.value_counts(normalize=True).min():.1%}")
    check(7, "Max Correlation < 0.95", abs(pd.concat([X,y],axis=1).corr()['Energy_Poverty']).drop('Energy_Poverty').max() < 0.95, "Safe")
    check(8, "No Direct Leakage in X", "Electricity" not in X.columns, "Clean")
    
    best_acc = max([r[1] for r in results])
    check(9, "Best Model Accuracy > 60%", best_acc > 0.60, f"{best_acc:.2%}")
    check(10, "Best Model Accuracy > 65%", best_acc > 0.65, f"{best_acc:.2%}")
    check(11, "At Least 3 Models Trained", len(results) >= 3, f"{len(results)} models")

    # Overfit checks
    m_rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42).fit(X_train, y_train)
    tr = accuracy_score(y_train, m_rf.predict(X_train))
    te = accuracy_score(y_test, m_rf.predict(X_test))
    check(12, "Random Forest Overfit Gap < 20%", (tr-te) < 0.20, f"gap={(tr-te):.2%}")
    
    m_lr = LogisticRegression(max_iter=500).fit(X_train, y_train)
    tr_l = accuracy_score(y_train, m_lr.predict(X_train))
    te_l = accuracy_score(y_test, m_lr.predict(X_test))
    check(13, "Logistic Regression Overfit Gap < 20%", (tr_l-te_l) < 0.20, f"gap={(tr_l-te_l):.2%}")
    check(14, "No Model Underfits (train > 55%)", tr > 0.55, f"train={tr:.2%}")

    # Stability
    cv_sc = cross_val_score(RandomForestClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5)
    check(15, "5-Fold CV Mean > 60%", cv_sc.mean() > 0.60, f"mean={cv_sc.mean():.2%}")
    check(16, "5-Fold CV Std Dev < 10%", cv_sc.std() < 0.10, f"std={cv_sc.std():.2%}")
    
    y_r = pd.Series(y).sample(frac=1, random_state=42)
    sh_acc = cross_val_score(DecisionTreeClassifier(max_depth=3), X, y_r, cv=3).mean()
    check(17, "Target Shuffle Baseline <= 75%", sh_acc <= 0.75, f"shuffle={sh_acc:.2%}")
    
    X_adv = X.drop([c for c in ['Electricity','hrs_elect','cfuel','avl_lpg','ipollut','rbiomass','dailycook'] if c in X.columns], axis=1)
    ad_acc = cross_val_score(DecisionTreeClassifier(max_depth=3), X_adv, y, cv=3).mean()
    check(18, "Adversarial Accuracy > 55%", ad_acc > 0.55, f"adv={ad_acc:.2%}")

    print(f"\nFINAL TEST STATUS: {passed}/18 PASSED")
    
    # Show Checklist
    CPATH = "MODEL_UNIT_TEST_CHECKLIST.txt"
    if os.path.exists(CPATH):
        print("\nFETCHING QA TEAM CHECKLIST:\n" + "="*70)
        with open(CPATH, "r") as f:
            print(f.read())

run_safe(stress_testing_suite, "Final QA Verification")


Executing: Final QA Verification
 SECTION 15: AUTOMATED QA STRESS TESTS 

[Pass] Test 01: No Missing Features (0 nulls)
[Pass] Test 02: No Missing Target (0 nulls)
[Pass] Test 03: Rows > 400 (499 rows)
[Pass] Test 04: Features > 20 (48 features)
[Pass] Test 05: Target Binary (0,1)
[Pass] Test 06: Minority >= 15% (46.5%)
[Pass] Test 07: Max Correlation < 0.95 (Safe)
[Pass] Test 08: No Direct Leakage in X (Clean)
[Pass] Test 09: Best Model Accuracy > 60% (66.67%)
[Pass] Test 10: Best Model Accuracy > 65% (66.67%)
[Pass] Test 11: At Least 3 Models Trained (4 models)
[Pass] Test 12: Random Forest Overfit Gap < 20% (gap=4.97%)
[Pass] Test 13: Logistic Regression Overfit Gap < 20% (gap=10.20%)
[Pass] Test 14: No Model Underfits (train > 55%) (train=71.63%)
[Pass] Test 15: 5-Fold CV Mean > 60% (mean=64.20%)
[Pass] Test 16: 5-Fold CV Std Dev < 10% (std=7.15%)
[Pass] Test 17: Target Shuffle Baseline <= 75% (shuffle=47.68%)
[Pass] Test 18: Adversarial Accuracy > 55% (adv=55.32%)

FINAL TEST STAT